# Chapter 11: SUT Abstraction — Adapters, Targets and Traces

**Learning objectives:**
- Understand the SUT adapter pattern and why abstraction matters for black-box testing
- Create `TargetProfile` and `IsolationRequirements` objects that formally identify a system under test
- Capture `SUTTrace` objects with tool call records and latency metadata
- Use `AgentAdapter` and `RAGAdapter` to wrap different SUT types behind a uniform interface
- Run a full 640-question evaluation through the adapter layer

---
## Preamble

Parts I–III built the full verification pipeline: ingestion, question generation, geometric judging, statistical analysis, claims, verifiers, verdicts and severity classification. Every experiment used a single evaluator function — `create_mock_evaluator()` — passed directly to `DOEGMSBenchmark.run()`.

That approach works for quick prototyping, but real-world testing must handle diverse systems: a bare LLM behind an API, a tool-using agent with recovery logic, a RAG pipeline with a retriever, or even a browser-based UI. Part IV introduces the **black-box testing layer** that wraps any of these behind a uniform interface.

This chapter introduces three abstractions that make that possible:

| Abstraction | Purpose |
|---|---|
| `TargetProfile` | Formal identity — who/what is being tested |
| `SUTAdapter` | Uniform evaluation interface — same `evaluate()` for every SUT type |
| `SUTTrace` | Structured execution trace — raw output plus tool calls, latency and token counts |

In [ ]:
# === Preamble (all notebooks) ===
import os
import warnings
warnings.filterwarnings("ignore")

import torch
import numpy as np
from collections import Counter

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

In [ ]:
# [AUTO-INJECTED] Disable DOE tracing for parallel execution safety
from knowlytix.harness.testing import DOEHarnessConfig
DOEHarnessConfig.__dataclass_fields__['enable_tracing'].default = False


In [ ]:
from importlib.resources import files
# Load the Chapter 4 store and set up the benchmark
from knowlytix.harness.testing import DOEGMSBenchmark, DOEHarnessConfig

config = DOEHarnessConfig(
    markdown_path=str(files("knowlytix.harness.testing").joinpath("tests/model_risk_assessment.md")),
    stores_dir="../lab_stores",
    store_name="ch04_ground_truth",
    n_runs=4,
    ingest_epochs=200,
    device=str(DEVICE),
)
bench = DOEGMSBenchmark(config)
store = bench.ingest()
print(f"Store: {store.stats()['entities']} entities, "
      f"{store.stats()['triples']} triples, "
      f"{store.stats()['enm_entries']} ENM")

---

## Why Abstraction Matters

In Parts I–III every experiment called `bench.run(evaluator=fn)` where `fn` was a simple `(question, context) -> answer` callable. That interface is sufficient when the only thing you need is the answer text — but black-box agentic testing requires much more:

| What we need | Why |
|---|---|
| **Tool call records** | To verify the agent used the right tools in the right order |
| **Latency per call** | To enforce SLA budgets and detect timeout risks |
| **Token counts** | To track cost and detect prompt-bloat regressions |
| **Retrieved context** | To audit RAG pipelines for retrieval quality |
| **Error capture** | To distinguish agent failures from test infrastructure failures |
| **Formal SUT identity** | To tie results to a specific version, model and configuration |

The adapter pattern solves this: every SUT type implements `evaluate(question, context) -> SUTTrace`, and the rest of the pipeline sees only traces.

---

## TargetProfile and IsolationRequirements

Before evaluating a system we must formally identify it. `TargetProfile` captures the SUT's identity, capabilities and resource constraints. `IsolationRequirements` specifies sandbox and safety constraints.

**TargetProfile fields:**

| Field | Type | Purpose |
|---|---|---|
| `target_id` | str | Unique ID (auto-generated as `tgt_<uuid8>` if empty) |
| `name` | str | Human-readable label |
| `version` | str | Semantic version or commit hash |
| `target_type` | str | One of `llm`, `agent`, `rag`, `browser` |
| `endpoint` | str | API URL or `"local"` for in-process |
| `model_name` | str | Foundation model name |
| `capabilities` | list | Declared capabilities (`tool_use`, `multi_turn`, etc.) |
| `timeout_ms` | int | Per-invocation timeout |
| `max_concurrent` | int | Maximum parallel invocations |
| `isolation` | IsolationRequirements | Sandbox constraints |

**IsolationRequirements fields:**

| Field | Type | Default | Purpose |
|---|---|---|---|
| `network_isolated` | bool | False | Restrict to whitelisted endpoints |
| `sandbox_filesystem` | bool | False | Redirect file I/O to temp sandbox |
| `synthetic_data_only` | bool | False | Use only synthetic test data |
| `max_memory_mb` | int | 0 | Memory cap (0 = unlimited) |
| `max_wall_time_s` | int | 0 | Wall-clock cap per run (0 = unlimited) |

In [ ]:
# Create a TargetProfile for the MockHarness
from knowlytix.harness.testing.target import TargetProfile, IsolationRequirements

isolation = IsolationRequirements(
    synthetic_data_only=True,
    max_memory_mb=512,
    max_wall_time_s=300,
)
print("IsolationRequirements:")
print(f"  network_isolated  : {isolation.network_isolated}")
print(f"  sandbox_filesystem: {isolation.sandbox_filesystem}")
print(f"  synthetic_data_only: {isolation.synthetic_data_only}")
print(f"  max_memory_mb     : {isolation.max_memory_mb}")
print(f"  max_wall_time_s   : {isolation.max_wall_time_s}")

target = TargetProfile(
    target_id="mock-v1",
    name="MockAgent",
    version="1.0.0",
    target_type="agent",
    model_name="mock-agent-v1",
    capabilities=["tool_use", "recovery"],
    isolation=isolation,
)
print("\nTargetProfile:")
print(f"  target_id  : {target.target_id}")
print(f"  name       : {target.name}")
print(f"  version    : {target.version}")
print(f"  target_type: {target.target_type}")
print(f"  endpoint   : {target.endpoint}")
print(f"  model_name : {target.model_name}")
print(f"  capabilities: {target.capabilities}")
print(f"  timeout_ms : {target.timeout_ms}")
print(f"  created_at : {target.created_at}")
print(f"\nSummary: {target.summary()}")
print(f"has_capability('tool_use') : {target.has_capability('tool_use')}")
print(f"has_capability('streaming'): {target.has_capability('streaming')}")

In [5]:
# YAML round-trip: save and reload
import tempfile

yaml_path = tempfile.mktemp(suffix=".yaml")
target.to_yaml(yaml_path)

# Show the YAML content
with open(yaml_path) as f:
    print("=== Saved YAML ===")
    print(f.read())

# Reload and verify
reloaded = TargetProfile.from_yaml(yaml_path)
print(f"Reloaded target_id  : {reloaded.target_id}")
print(f"Reloaded name       : {reloaded.name}")
print(f"Reloaded isolation  : {reloaded.isolation.to_dict()}")
print(f"Round-trip match    : {reloaded.to_dict() == target.to_dict()}")
os.remove(yaml_path)

=== Saved YAML ===
target:
  target_id: mock-v1
  name: MockAgent
  version: 1.0.0
  target_type: agent
  endpoint: local
  model_name: mock-agent-v1
  capabilities:
  - tool_use
  - recovery
  timeout_ms: 60000
  max_concurrent: 1
  isolation:
    network_isolated: false
    sandbox_filesystem: false
    synthetic_data_only: true
    max_memory_mb: 512
    max_wall_time_s: 300
  metadata: {}
  created_at: '2026-04-15T21:48:20'

Reloaded target_id  : mock-v1
Reloaded name       : MockAgent
Reloaded isolation  : {'network_isolated': False, 'sandbox_filesystem': False, 'synthetic_data_only': True, 'max_memory_mb': 512, 'max_wall_time_s': 300}
Round-trip match    : True


---

## SUTTrace and ToolCallRecord

Every adapter call returns a `SUTTrace` — a structured record that carries the answer *and* all execution metadata needed for downstream verification.

**SUTTrace fields:**

| Field | Type | Purpose |
|---|---|---|
| `raw_output` | str | The agent's answer text |
| `tool_calls` | list[ToolCallRecord] | Every tool invocation during execution |
| `latency_ms` | float | Wall-clock time for the full evaluation |
| `token_count` | int | Total tokens (input + output) |
| `input_tokens` | int | Prompt tokens |
| `output_tokens` | int | Completion tokens |
| `retrieved_context` | str | Retrieved text (RAG adapters only) |
| `error` | str | Error message if execution failed |

**ToolCallRecord fields:**

| Field | Type | Purpose |
|---|---|---|
| `tool_name` | str | Name of the tool called |
| `arguments` | dict | Arguments passed to the tool |
| `result` | str | Tool result (truncated to 500 chars) |
| `latency_ms` | float | Time for this individual tool call |

The `tool_calls` list is especially important — Chapter 15 will use it for `TraceOracle` to verify tool ordering via holonomy.

In [ ]:
# Inspect the data models
from knowlytix.harness.testing.adapters import SUTTrace, ToolCallRecord
from dataclasses import fields

print("SUTTrace fields:")
for f in fields(SUTTrace):
    print(f"  {f.name:20s}  {f.type}")

print("\nToolCallRecord fields:")
for f in fields(ToolCallRecord):
    print(f"  {f.name:20s}  {f.type}")

---

## The SUTAdapter Hierarchy

`SUTAdapter` is an abstract base class with a single method: `evaluate(question, context) -> SUTTrace`. Three concrete adapters cover the main SUT types:

| Adapter | SUT type | Key behavior |
|---|---|---|
| `LLMAdapter` | Direct API call | Captures token counts from API response |
| `AgentAdapter` | Tool-using agent | Hooks into harness callbacks to capture every `ToolCallRecord` |
| `RAGAdapter` | Retriever + generator | Captures `retrieved_context` alongside the answer |

The `AgentAdapter` is the most complex — it installs `on_tool_call` and `on_tool_result` hooks on the harness to intercept every tool invocation transparently. The original hooks are saved and restored after each evaluation to support composition with `ToolGateway` (Chapter 12).

A factory function `make_adapter(target_type, target_model, ...)` selects the right adapter from a target type string.

---

## Lab 11.1 — Wrap the Mock Agent as a Proper SUT

In this lab we wrap the `MockHarness` with an `AgentAdapter` and evaluate a single question, then inspect every field of the resulting `SUTTrace`.

In [ ]:
from importlib.resources import files
# Lab 11.1a — Create the AgentAdapter
from knowlytix.harness.testing.adapters import AgentAdapter, SUTTrace
from knowlytix.harness.fixtures import MockHarness

harness = MockHarness(recovery_mode="fallback")
adapter = AgentAdapter(harness)
print(f"Adapter: {adapter}")

# Load context
context = open(str(files("knowlytix.harness.testing").joinpath("tests/model_risk_assessment.md"))).read()

# Evaluate a single question
trace = adapter.evaluate(
    question="What is the test AUC of MDL-001?",
    context=context,
)
print("\n=== SUTTrace ===")
print(f"raw_output : {trace.raw_output!r}")
print(f"latency_ms : {trace.latency_ms:.1f}")
print(f"token_count: {trace.token_count}")
print(f"error      : {trace.error!r}")
print(f"tool_calls : {len(trace.tool_calls)} calls")

In [8]:
# Lab 11.1b — Inspect tool call records
print("=== Tool Call Records ===")
for i, call in enumerate(trace.tool_calls):
    print(f"\nCall {i+1}:")
    print(f"  tool_name  : {call.tool_name}")
    print(f"  arguments  : {call.arguments}")
    print(f"  result     : {call.result[:120]}...")
    print(f"  latency_ms : {call.latency_ms:.2f}")

=== Tool Call Records ===

Call 1:
  tool_name  : data_lookup
  arguments  : {'entity': 'MDL-001'}
  result     : {'name': 'CreditScore-XGB', 'domain': 'Credit Decisioning', 'risk': 'High', 'status': 'Degraded', 'algorithm': 'XGBoost ...
  latency_ms : 0.01

Call 2:
  tool_name  : risk_check
  arguments  : {'model_id': 'MDL-001'}
  result     : Model MDL-001 (CreditScore-XGB): Risk=High, Status=Degraded, Domain=Credit Decisioning...
  latency_ms : 0.00


In [9]:
# Lab 11.1c — Test with a different question type (risk/compliance)
trace2 = adapter.evaluate(
    question="What is the risk rating of MDL-003?",
    context=context,
)
print("Question  : What is the risk rating of MDL-003?")
print(f"Answer    : {trace2.raw_output!r}")
print(f"Latency   : {trace2.latency_ms:.1f} ms")
print(f"Tool calls: {len(trace2.tool_calls)}")
for call in trace2.tool_calls:
    print(f"  -> {call.tool_name}({call.arguments})")

Question  : What is the risk rating of MDL-003?
Answer    : '0.8934'
Latency   : 0.4 ms
Tool calls: 2
  -> data_lookup({'entity': 'MDL-001'})
  -> risk_check({'model_id': 'MDL-001'})


In [10]:
# Lab 11.1d — Test with an escalation question
trace3 = adapter.evaluate(
    question="Should we escalate the review of MDL-001 given its degraded status?",
    context=context,
)
print("Question  : Should we escalate the review of MDL-001?")
print(f"Answer    : {trace3.raw_output[:200]!r}")
print(f"Tool calls: {len(trace3.tool_calls)}")
for call in trace3.tool_calls:
    print(f"  -> {call.tool_name}({call.arguments})")

Question  : Should we escalate the review of MDL-001?
Answer    : '0.8934'
Tool calls: 2
  -> data_lookup({'entity': 'MDL-001'})
  -> risk_check({'model_id': 'MDL-001'})


---

## Lab 11.2 — Adapter Comparison: AgentAdapter vs RAGAdapter

The `RAGAdapter` wraps a two-stage pipeline: a `retriever` callable that fetches context and a `generator` callable that produces the answer. Unlike `AgentAdapter`, it captures `retrieved_context` instead of `tool_calls`. We build a simple mock RAG pipeline to demonstrate the difference.

In [ ]:
# Lab 11.2a — Build a mock RAG pipeline
from knowlytix.harness.testing.adapters import RAGAdapter
import re

def mock_retriever(question: str, context: str) -> str:
    """Simple keyword retriever: find lines matching question entities."""
    keywords = re.findall(r"MDL-\d{3}|[A-Z]{2,}", question)
    lines = context.split("\n")
    hits = [l for l in lines if any(k in l for k in keywords)]
    return "\n".join(hits[:10]) if hits else lines[0]

def mock_generator(question: str, retrieved: str) -> str:
    """Simple extractor: pull numbers or key phrases from retrieved text."""
    numbers = re.findall(r"\d+\.\d+", retrieved)
    if numbers:
        return numbers[0]
    return retrieved[:200]

rag_adapter = RAGAdapter(retriever=mock_retriever, generator=mock_generator)
print(f"RAGAdapter: {rag_adapter}")

rag_trace = rag_adapter.evaluate(
    question="What is the test AUC of MDL-001?",
    context=context,
)
print("\n=== RAG SUTTrace ===")
print(f"raw_output        : {rag_trace.raw_output!r}")
print(f"latency_ms        : {rag_trace.latency_ms:.1f}")
print(f"tool_calls        : {len(rag_trace.tool_calls)} (always 0 for RAG)")
print(f"retrieved_context : {rag_trace.retrieved_context[:200]!r}...")
print(f"error             : {rag_trace.error!r}")

In [12]:
# Lab 11.2b — Side-by-side comparison
print("=== Adapter Comparison (same question) ===")
print(f"{'Field':<22s} {'AgentAdapter':<30s} {'RAGAdapter':<30s}")
print("-" * 82)
print(f"{'raw_output':<22s} {trace.raw_output[:25]!r:<30s} {rag_trace.raw_output[:25]!r:<30s}")
print(f"{'latency_ms':<22s} {trace.latency_ms:<30.1f} {rag_trace.latency_ms:<30.1f}")
print(f"{'tool_calls':<22s} {len(trace.tool_calls):<30d} {len(rag_trace.tool_calls):<30d}")
print(f"{'retrieved_context':<22s} {'(not captured)':<30s} {len(rag_trace.retrieved_context):>4d} chars")
print(f"{'token_count':<22s} {trace.token_count:<30d} {rag_trace.token_count:<30d}")

=== Adapter Comparison (same question) ===
Field                  AgentAdapter                   RAGAdapter                    
----------------------------------------------------------------------------------
raw_output             '0.8934'                       '0.8934'                      
latency_ms             0.6                            0.1                           
tool_calls             2                              0                             
retrieved_context      (not captured)                  669 chars
token_count            0                              0                             


In [ ]:
# Lab 11.2c — Factory function: make_adapter
from knowlytix.harness.testing.adapters import make_adapter

agent_adapter = make_adapter(
    target_type="agent",
    target_model="mock-agent-v1",
    harness=MockHarness(recovery_mode="fallback"),
)
print(f"make_adapter('agent', ...) -> {agent_adapter}")

# LLMAdapter requires an API client — show the error for pedagogical purposes
try:
    llm_adapter = make_adapter(target_type="llm", target_model="claude-sonnet-4-20250514")
    print(f"make_adapter('llm', ...)   -> {llm_adapter}")
except Exception as e:
    print(f"make_adapter('llm', ...)   -> (requires API key: {type(e).__name__})")

# Unknown type
try:
    make_adapter(target_type="unknown", target_model="x")
except ValueError as e:
    print(f"make_adapter('unknown')    -> ValueError: {e}")

---

## End-to-End: 640 Questions Through the Adapter

Now we run the full 640-question benchmark through the `AgentAdapter` to collect traces for every question. This demonstrates the adapter as the entry point for systematic evaluation — every trace carries the answer plus full execution metadata.

In [ ]:
from importlib.resources import files
# Run all 640 DOE questions through the AgentAdapter
adapter = AgentAdapter(MockHarness(recovery_mode="fallback"))
context = open(str(files("knowlytix.harness.testing").joinpath("tests/model_risk_assessment.md"))).read()

doe_questions = bench.generate_questions()
print(f"Generated {len(doe_questions)} DOE questions")

traces = []
for dq in doe_questions:
    t = adapter.evaluate(question=dq.question.natural_language, context=context)
    traces.append(t)

print(f"Collected {len(traces)} traces")
print(f"Errors   : {sum(1 for t in traces if t.error)}")
print(f"Non-empty: {sum(1 for t in traces if t.raw_output)}")

In [15]:
# Latency analysis
latencies = [t.latency_ms for t in traces]
print("=== Latency Summary ===")
print(f"Mean   : {np.mean(latencies):.2f} ms")
print(f"Median : {np.median(latencies):.2f} ms")
print(f"Std    : {np.std(latencies):.2f} ms")
print(f"Min    : {np.min(latencies):.2f} ms")
print(f"Max    : {np.max(latencies):.2f} ms")
print(f"P95    : {np.percentile(latencies, 95):.2f} ms")
print(f"P99    : {np.percentile(latencies, 99):.2f} ms")

=== Latency Summary ===
Mean   : 0.36 ms
Median : 0.36 ms
Std    : 0.02 ms
Min    : 0.36 ms
Max    : 0.92 ms
P95    : 0.36 ms
P99    : 0.37 ms


In [16]:
# Tool call analysis
all_tool_calls = [call for t in traces for call in t.tool_calls]
print("=== Tool Call Summary ===")
print(f"Total tool calls : {len(all_tool_calls)}")
print(f"Calls per trace  : mean={np.mean([len(t.tool_calls) for t in traces]):.1f}, "
      f"min={min(len(t.tool_calls) for t in traces)}, "
      f"max={max(len(t.tool_calls) for t in traces)}")

# Tool frequency
tool_counts = Counter(c.tool_name for c in all_tool_calls)
print("\nTool frequency:")
for tool, count in tool_counts.most_common():
    print(f"  {tool:<22s} {count:>4d}  ({100*count/len(all_tool_calls):.1f}%)")

# Tool latency by tool name
print("\nMean latency per tool:")
for tool in sorted(tool_counts.keys()):
    tool_lats = [c.latency_ms for c in all_tool_calls if c.tool_name == tool]
    print(f"  {tool:<22s} {np.mean(tool_lats):.3f} ms")

=== Tool Call Summary ===
Total tool calls : 1280
Calls per trace  : mean=2.0, min=2, max=2

Tool frequency:
  data_lookup             640  (50.0%)
  risk_check              640  (50.0%)

Mean latency per tool:
  data_lookup            0.002 ms
  risk_check             0.001 ms


In [17]:
# Tool calls by question category
print("=== Tool Calls by Question Category ===")
cat_tools = {}
for dq, t in zip(doe_questions, traces):
    cat = dq.question.category
    if cat not in cat_tools:
        cat_tools[cat] = {"count": 0, "total_calls": 0, "tools": Counter()}
    cat_tools[cat]["count"] += 1
    cat_tools[cat]["total_calls"] += len(t.tool_calls)
    for call in t.tool_calls:
        cat_tools[cat]["tools"][call.tool_name] += 1

print(f"{'Category':<28s} {'Qs':>4s} {'Calls':>6s} {'Calls/Q':>8s}  Top Tool")
print("-" * 80)
for cat in sorted(cat_tools.keys()):
    info = cat_tools[cat]
    avg = info["total_calls"] / info["count"] if info["count"] else 0
    top = info["tools"].most_common(1)[0] if info["tools"] else ("none", 0)
    print(f"{cat:<28s} {info['count']:>4d} {info['total_calls']:>6d} {avg:>8.1f}  {top[0]} ({top[1]})")

=== Tool Calls by Question Category ===
Category                       Qs  Calls  Calls/Q  Top Tool
--------------------------------------------------------------------------------
adversarial                    40     80      2.0  data_lookup (40)
boundary                       40     80      2.0  data_lookup (40)
counting                       40     80      2.0  data_lookup (40)
cross_reference                40     80      2.0  data_lookup (40)
exact_retrieval                40     80      2.0  data_lookup (40)
governance                     40     80      2.0  data_lookup (40)
holonomy                       40     80      2.0  data_lookup (40)
link_prediction                40     80      2.0  data_lookup (40)
multi_hop                      40     80      2.0  data_lookup (40)
multi_hop_transport            40     80      2.0  data_lookup (40)
phase_comparison               40     80      2.0  data_lookup (40)
plausibility                   40     80      2.0  data_lookup (40)
rec

In [18]:
# Answer length distribution
answer_lens = [len(t.raw_output) for t in traces]
print("=== Answer Length Summary ===")
print(f"Mean   : {np.mean(answer_lens):.1f} chars")
print(f"Median : {np.median(answer_lens):.1f} chars")
print(f"Min    : {np.min(answer_lens)} chars")
print(f"Max    : {np.max(answer_lens)} chars")
print(f"Empty  : {sum(1 for l in answer_lens if l == 0)}")

=== Answer Length Summary ===
Mean   : 6.0 chars
Median : 6.0 chars
Min    : 6 chars
Max    : 6 chars
Empty  : 0


---

## Connecting the Adapter to the Verification Pipeline

The adapter slot fits naturally into the pipeline from Parts I–III. Instead of passing a raw evaluator function, we wrap the adapter in a callable that extracts `raw_output` from the trace. This preserves backward compatibility while enabling trace capture.

In [ ]:
# Connect adapter traces to the verification pipeline (Ch 8–10)
from knowlytix.harness.testing.claims import ClaimCompiler
from knowlytix.harness.testing.verifiers import VerificationRouter
from knowlytix.harness.testing.verdict import TypedVerdict

compiler = ClaimCompiler(store=store)
router = VerificationRouter(store=store)

# Pick 5 representative traces and run them through the pipeline
sample_indices = [0, 100, 200, 300, 400]
print("=== Adapter -> Claim -> Verdict Pipeline (5 samples) ===\n")

for idx in sample_indices:
    dq = doe_questions[idx]
    t = traces[idx]

    claims = compiler.compile(
        llm_answer=t.raw_output,
        question_metadata=dq.question.metadata,
        question_category=dq.question.category,
        run_id="ch11-demo",
        question_id=f"q-{idx:03d}",
        ground_truth=str(dq.question.ground_truth),
        question_text=dq.question.natural_language,
    )
    sub_verdicts = router.route_all(claims)
    verdict = TypedVerdict.from_sub_verdicts(
        claims=claims,
        sub_verdicts=sub_verdicts,
        exact_correct=True,
        exact_detail="ground truth",
        run_id="ch11-demo",
        question_id=f"q-{idx:03d}",
    )

    passed = sum(1 for sv in sub_verdicts if sv.passed)
    print(f"Q{idx:>3d} [{dq.question.category}]")
    print(f"  Answer     : {t.raw_output[:60]!r}")
    print(f"  Tool calls : {len(t.tool_calls)}")
    print(f"  Claims     : {len(claims)}, verified {passed}/{len(sub_verdicts)}")
    print(f"  Confidence : {verdict.overall_confidence:.4f}")
    print()

---

## Exercises

**Exercise 11.1** — Create a `TargetProfile` for a hypothetical production agent with `target_type="agent"`, `model_name="claude-sonnet-4-20250514"`, capabilities `["tool_use", "multi_turn", "file_upload"]` and isolation constraints `network_isolated=True`, `max_wall_time_s=120`. Save it to YAML and reload it. Verify the round-trip preserves all fields.

**Exercise 11.2** — Modify the `mock_retriever` in Lab 11.2 to use GMS `search()` instead of keyword matching. Compare the `retrieved_context` length and the answer quality against the keyword retriever.

**Exercise 11.3** — Run the 640-question benchmark through the `RAGAdapter` (using the mock retriever/generator from Lab 11.2) and compare the pass rate, latency distribution and answer length distribution against the `AgentAdapter` results. Which adapter type performs better on which question categories?

---

## Summary

This chapter introduced the three abstractions that underpin Part IV:

1. **TargetProfile** formally identifies a SUT with its type, capabilities, isolation constraints and metadata. Profiles serialize to YAML for reproducible test configurations.

2. **SUTAdapter** provides a uniform `evaluate()` interface across SUT types — `AgentAdapter` captures tool call records, `RAGAdapter` captures retrieved context and `LLMAdapter` captures token counts.

3. **SUTTrace** is the structured execution record that flows through the rest of the pipeline: answer text, tool calls, latency and error status.

In the next chapter we install a **ToolGateway** on the adapter to inject faults, block tools and observe recovery behavior — turning the adapter into a full black-box testing harness.